# 🤖 Agentic RAG Workshop — 4 ชั่วโมงเดียวจบ

**Agentic RAG: From Zero to Hero** | Workshop 4 ชม.

---

### 🎯 จุดประสงค์การเรียนรู้

เมื่อจบ workshop ผู้เรียนสามารถ:
1. **เตรียมข้อมูลสำหรับ RAG** — Chunking + Embedding + VectorDB
2. **สร้าง AI Agent** — ใช้ Google ADK สร้าง Agent + Custom Tool
3. **สร้าง Agentic RAG** ⭐ — Agent ที่ค้นหาจาก VectorDB + ตัดสินใจ + ตอบคำถามเอง
4. **วัดคุณภาพ** — ใช้ LLM-as-Judge ประเมิน Agent

### ⏱️ Timeline

| Part | เวลา | เนื้อหา |
|------|:----:|--------|
| 📢 Part 1 | 1 ชม. 20 นาที | Data → Chunk → Embed → Qdrant → Retrieve |
| ☕ พัก | 10 นาที | |
| 📢 Part 2 | 1 ชม. 30 นาที | Agent → Tool → RAG Agent → Agentic RAG |
| 🧪 Part 3 | 1 ชม. | แบบฝึกหัด + Q&A |

---
## 📢 Part 1: Data → VectorDB

### RAG คืออะไร?

**R**etrieval-**A**ugmented **G**eneration = ค้นหา + สร้างคำตอบ

```
LLM เก่ง แต่ไม่รู้ข้อมูลเรา
→ ต้อง "ค้นหา" ข้อมูลที่เกี่ยวข้องมาให้ก่อน
→ แล้ว LLM จะ "สร้าง" คำตอบจากข้อมูลนั้น
```

| Pipeline วันนี้ |
|---|
| 📄 Text → ✂️ Chunking → 🔢 Embedding → 🗄️ Qdrant → 🔎 Retrieve → 🤖 Agent → 💬 Answer |

## 📦 Section 0: Setup

ติดตั้ง libraries ทั้งหมดที่จำเป็น

In [ ]:
%%time
# ติดตั้ง libraries
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['google-adk', 'google-genai', 'sentence-transformers', 'qdrant-client', 'langchain-text-splitters', 'scikit-learn']:
    _pip_install(_pkg)

print('✅ ติดตั้งเรียบร้อย!')
print('⚠️ หลัง install ครั้งแรก ให้ Restart runtime: Runtime → Restart session → แล้ว Run ใหม่ตั้งแต่ cell ถัดไป')

In [ ]:
%%time
# ตั้งค่า Gemini API Key
import os

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ โหลด API Key จาก Colab Secrets')
except Exception:
    os.environ['GOOGLE_API_KEY'] = input('🔑 กรุณาวาง Gemini API Key: ')

# ทดสอบ API
from google import genai
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
resp = client.models.generate_content(model='gemini-2.5-flash', contents='สวัสดีครับ ตอบสั้นๆ 1 ประโยค')
print(f'🤖 Gemini: {resp.text}')

### 📄 เตรียมข้อมูลตัวอย่าง

สร้างข้อมูล Case Study ภาษาไทยสำหรับใช้ตลอด workshop

In [ ]:
%%time
# สร้างข้อมูลตัวอย่าง
import os
os.makedirs('data', exist_ok=True)

sample_texts = {
    'case_ict_mahidol.txt': '''กรณีศึกษา: AI ที่คณะ ICT มหาวิทยาลัยมหิดล

คณะเทคโนโลยีสารสนเทศและการสื่อสาร (ICT) มหาวิทยาลัยมหิดล
เป็นที่ตั้งของ Mahidol AI Center สถาบันปัญญาประดิษฐ์มหิดล
มีคลัสเตอร์วิจัย MIKE (Machine Intelligence and Knowledge Engineering)

มุ่งเน้นงานวิจัย AI ด้านการแพทย์ จีโนมิกส์ และ Federated Learning
โครงการเด่น: Siribot แชทบอทสำหรับโรงพยาบาลศิริราช
AI Medical Imaging วิเคราะห์ภาพ X-Ray และ VIS4ION ระบบนำทางผู้พิการทางสายตา

คณะ ICT ยังนำ RAG มาสร้างระบบ AI Tutor
ช่วยตอบคำถามวิชาเรียนจากเอกสารประกอบการสอน
นักศึกษาถามคำถามได้ตลอด 24 ชั่วโมง โดยค้นคำตอบจาก lecture notes slides และตำราเรียน''',

    'case_healthcare.txt': '''กรณีศึกษา: AI ในการแพทย์ไทย

โรงพยาบาลศิริราชนำ AI มาวิเคราะห์ภาพถ่ายทางการแพทย์
เช่น การตรวจจับมะเร็งปอดจากภาพ X-ray ด้วย Deep Learning
ผลการทดสอบพบว่า AI มีความแม่นยำ 95%

มีการใช้ NLP วิเคราะห์เวชระเบียนอิเล็กทรอนิกส์
ลดเวลาการอ่านเวชระเบียนจาก 15 นาทีเหลือ 2 นาทีต่อเคส

ความท้าทาย: ข้อมูลทางการแพทย์ภาษาไทยมีจำนวนจำกัด
ต้องใช้ Transfer Learning จากโมเดลภาษาอังกฤษ''',

    'case_banking.txt': '''กรณีศึกษา: AI ในธนาคารและการเงิน

ธนาคารกสิกรไทยพัฒนาระบบ Chatbot ที่ใช้ LLM ร่วมกับ RAG
ตอบคำถามลูกค้าเกี่ยวกับผลิตภัณฑ์ทางการเงิน

ระบบ RAG ค้นหาข้อมูลจากฐานความรู้ภายใน
ประกอบด้วยเอกสารผลิตภัณฑ์ เงื่อนไขบริการ และ FAQ
LLM สร้างคำตอบที่เป็นธรรมชาติจากข้อมูลที่ค้นพบ

ผลลัพธ์: ลดภาระ Call Center 40%
ความพึงพอใจลูกค้าเพิ่มขึ้น 25%
ใช้ Vector Database เก็บ Embedding + Hybrid Search''',

    'case_agriculture.txt': '''กรณีศึกษา: AI ในการเกษตรอัจฉริยะ

Smart Farming ในไทยใช้ AI วิเคราะห์ภาพถ่ายดาวเทียม
และข้อมูลจาก IoT Sensor เพื่อพยากรณ์ผลผลิตและเฝ้าระวังโรคพืช

ระบบ Computer Vision ตรวจจับโรคข้าวจากภาพถ่ายใบข้าว
ใช้ CNN ที่ Train ด้วยภาพถ่ายโรคข้าวกว่า 50,000 ภาพ

มีการใช้ NLP วิเคราะห์ข้อมูลราคาสินค้าเกษตร
จากข่าวสาร รายงานภาครัฐ และ Social Media
ช่วยเกษตรกรตัดสินใจเรื่องการปลูกและการขาย'''
}

for fname, content in sample_texts.items():
    with open(f'data/{fname}', 'w', encoding='utf-8') as f:
        f.write(content)

print(f'✅ สร้างไฟล์ตัวอย่าง {len(sample_texts)} ไฟล์')
for fname in sorted(sample_texts.keys()):
    print(f'  📄 {fname}')

---
## ✂️ Section 1: Chunking — ตัดข้อความเป็นส่วนย่อย

### ทำไมต้อง Chunk?

- LLM มี **context window จำกัด** — ส่งเอกสาร 100 หน้าทั้งหมดไม่ได้
- Embedding model ทำงานดีกับข้อความ **สั้นถึงปานกลาง**
- Chunk ที่เหมาะสมช่วยให้ **retrieval แม่นยำขึ้น**

| วิธี | หลักการ | ข้อดี | ข้อเสีย |
|------|---------|-------|--------|
| **Fixed-size** | ตัดตามจำนวนตัวอักษร | ง่าย, เร็ว | อาจตัดกลางประโยค |
| **Recursive** ⭐ | แบ่งตาม separator ซ้อนกัน | ผลดี, รักษาโครงสร้าง | ต้องกำหนด separators |

In [ ]:
%%time
# โหลดข้อความจากไฟล์
all_text = ''
for fname in sorted(os.listdir('data')):
    fpath = f'data/{fname}'
    if os.path.isfile(fpath) and fname.endswith('.txt'):
        with open(fpath, 'r', encoding='utf-8') as f:
            all_text += '\n\n' + f.read()

print(f'📄 ข้อความรวม: {len(all_text)} ตัวอักษร')

In [ ]:
%%time
# ─── Fixed-size Chunking ───
def fixed_chunk(text, size=200, overlap=30):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])
    return chunks

fixed_chunks = fixed_chunk(all_text, size=200, overlap=30)
print(f'📊 Fixed-size: ได้ {len(fixed_chunks)} chunks')
print(f'\n--- Chunk ที่ 1 (สังเกต: อาจตัดกลางคำ) ---')
print(fixed_chunks[0][:150] + '...')

In [ ]:
%%time
# ─── Recursive Chunking ⭐ (แนะนำ) ───
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=['\n\n', '\n', ' ', '']  # ลำดับ: ย่อหน้า → บรรทัด → คำ → ตัวอักษร
)

chunks = splitter.split_text(all_text)
print(f'📊 Recursive: ได้ {len(chunks)} chunks')
print(f'\n--- Chunk ที่ 1 (สังเกต: ตัดตามย่อหน้า) ---')
print(chunks[0])
print(f'\n--- Chunk สุดท้าย ---')
print(chunks[-1])

### 💡 สังเกต
- **Fixed-size** ตัดง่ายแต่อาจตัดกลางคำ/ประโยค
- **Recursive** ลองตัดตาม `\n\n` ก่อน ถ้ายังใหญ่ก็ตาม `\n` → ดีกว่ามาก
- `chunk_size` เล็ก → ชิ้นเล็กแม่นยำ แต่อาจขาด context
- `chunk_size` ใหญ่ → มี context แต่อาจมี noise

> 🎯 ใน workshop นี้เราจะใช้ **Recursive Chunking** ต่อไป

---
## 🔢 Section 2: Embedding — แปลงข้อความเป็น Vector

### Embedding คืออะไร?

แปลงข้อความ → ชุดตัวเลข (Vector) ที่แสดงความหมาย

```
"แมว" → [0.21, -0.85, 0.44, ...] (1024 มิติ)
"สุนัข" → [0.19, -0.82, 0.41, ...]  ← ใกล้เคียง "แมว" (สัตว์เลี้ยงเหมือนกัน)
"รถยนต์" → [-0.55, 0.31, -0.12, ...] ← ห่างจาก "แมว" (ไม่เกี่ยวกัน)
```

**Model ที่ใช้:** `intfloat/multilingual-e5-large` — รองรับภาษาไทย

> ⚠️ ต้องใส่ prefix: `'query: '` สำหรับคำถาม, `'passage: '` สำหรับเนื้อหา

In [ ]:
%%time
# โหลด Embedding Model (ครั้งแรกจะดาวน์โหลด ~2 นาที)
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
print(f'✅ โหลด model สำเร็จ | Vector size: {embed_model.get_sentence_embedding_dimension()}')

In [ ]:
%%time
# สร้าง embedding ของทุก chunk
passages = ['passage: ' + c for c in chunks]
chunk_embeddings = embed_model.encode(passages, show_progress_bar=True)

print(f'✅ Embed {len(chunks)} chunks สำเร็จ')
print(f'📐 Shape: {chunk_embeddings.shape}')

In [ ]:
%%time
# ทดลอง Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = 'query: ระบบ AI ช่วยตอบคำถามนักศึกษา'
query_emb = embed_model.encode(query)

# คำนวณ similarity กับทุก chunk
scores = cosine_similarity([query_emb], chunk_embeddings)[0]
top_idx = np.argsort(scores)[::-1][:3]

print(f'🔍 Query: "{query}"')
print(f'\n📊 Top-3 chunks ที่คล้ายสุด:')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank}. [score={scores[idx]:.4f}] {chunks[idx][:80]}...')

### 💡 สังเกต
- ข้อความที่มีความหมาย **ใกล้เคียง** → cosine similarity **สูง** (ใกล้ 1.0)
- ต้องใส่ prefix `query:` / `passage:` ตามที่ model กำหนด
- Embedding เข้าใจ **ความหมาย** ไม่ใช่แค่ keyword ตรงๆ

> 🎯 ต่อไปเราจะเก็บ embedding เหล่านี้ลง **Vector Database** เพื่อค้นหาได้เร็วขึ้น

---
## 🗄️ Section 3: Qdrant — Vector Database + Retrieve

### ทำไมไม่ใช้ DB ปกติ?

- DB ปกติ (SQL): ค้นหาด้วย keyword / exact match
- VectorDB: ค้นหาด้วย **ความหมาย** (semantic search)

**Qdrant** = VectorDB ที่ใช้ง่าย, เร็ว, รองรับ in-memory

In [ ]:
%%time
# สร้าง Qdrant Client + Collection
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(':memory:')  # ใช้ RAM (ไม่ต้องติดตั้ง server)

COLLECTION = 'workshop_docs'
VECTOR_SIZE = chunk_embeddings.shape[1]  # 1024

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

print(f'✅ สร้าง Collection "{COLLECTION}" สำเร็จ (vector size={VECTOR_SIZE})')

In [ ]:
%%time
# Upsert ทุก chunk เข้า Qdrant
points = [
    models.PointStruct(
        id=i,
        vector=chunk_embeddings[i].tolist(),
        payload={'text': chunks[i], 'chunk_id': i}
    )
    for i in range(len(chunks))
]

qdrant.upsert(collection_name=COLLECTION, points=points)
print(f'✅ Upsert {len(points)} chunks เข้า Qdrant สำเร็จ!')

In [ ]:
%%time
# ─── ค้นหาจาก Qdrant ───
def search_qdrant(query_text, top_k=3):
    """ค้นหา chunks จาก Qdrant ด้วย query"""
    q_emb = embed_model.encode(f'query: {query_text}')
    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_emb.tolist(),
        limit=top_k
    ).points
    return results

# ทดลองค้นหา
results = search_qdrant('Mahidol AI Center งานวิจัยปัญญาประดิษฐ์')
print('🔍 Query: "Mahidol AI Center งานวิจัยปัญญาประดิษฐ์"')
print(f'\n📊 Top-{len(results)} ผลลัพธ์จาก Qdrant:')
for i, r in enumerate(results, 1):
    print(f'  {i}. [score={r.score:.4f}] {r.payload["text"][:80]}...')

### 💡 สังเกต
- Qdrant ค้นหาด้วย **vector similarity** → เข้าใจความหมาย ไม่ใช่แค่ keyword
- ถามเรื่อง "แพทย์" → ได้ผลเกี่ยวกับ case_healthcare.txt
- `top_k` ควบคุมจำนวนผลลัพธ์ที่ส่งให้ LLM

> 🎯 ตอนนี้เรามี **RAG Pipeline ครบแล้ว!** ต่อไปเราจะสร้าง **Agent** ที่ใช้ pipeline นี้

---
### ☕ พัก 10 นาที

---

## 📢 Part 2: Agent + Agentic RAG

---
## 🤖 Section 4: Agent แรกของคุณ — Google ADK

### Agent vs Chatbot

🤔 **นึกภาพ:** คุณอยากจองตั๋วเครื่องบิน
- **Chatbot**: "กรุณาเลือก 1. ดูเที่ยวบิน 2. เช็คราคา 3. ติดต่อเจ้าหน้าที่" ← ทำได้แค่ตาม script
- **Agent**: "หาเที่ยวบิน กรุงเทพ→เชียงใหม่ วันศุกร์ ราคาถูกสุด จองเลย" ← **คิดเอง + ใช้เครื่องมือเอง**

| | Chatbot | Agent |
|---|---------|-------|
| ทำงาน | ถาม-ตอบตาม script | **คิด → ตัดสินใจ → ใช้เครื่องมือ** |
| ความยืดหยุ่น | ต่ำ (ต้อง hard-code) | สูง (ปรับตัวได้) |
| ตัวอย่าง | FAQ bot, เมนูโทรศัพท์ | AI จองตั๋ว, AI ค้นข้อมูล, Agentic RAG |

**Google ADK** = Agent Development Kit สร้าง Agent ด้วย Python ไม่กี่บรรทัด

In [ ]:
%%time
# สร้าง Agent แรก
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# สร้าง Agent ที่เป็น "ผู้ช่วยอาจารย์"
teacher_agent = LlmAgent(
    name='teacher_assistant',
    model='gemini-2.5-flash',
    instruction='คุณเป็นผู้ช่วยอาจารย์มหาวิทยาลัย ตอบคำถามเกี่ยวกับ AI สั้นกระชับ เป็นภาษาไทย ไม่เกิน 3 ประโยค'
)

print(f'✅ สร้าง Agent "{teacher_agent.name}" สำเร็จ')

In [ ]:
# ─── คุยกับ Agent ───
import asyncio

async def chat_with_agent(agent, message):
    """ส่งข้อความไปหา Agent แล้วรับคำตอบ"""
    runner = InMemoryRunner(agent=agent, app_name='workshop')
    session = await runner.session_service.create_session(
        app_name='workshop', user_id='student'
    )
    
    content = genai_types.Content(
        role='user',
        parts=[genai_types.Part(text=message)]
    )
    
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    
    return response_text

# ถามคำถาม
answer = await chat_with_agent(teacher_agent, 'RAG คืออะไร?')
print(f'❓ คำถาม: RAG คืออะไร?')
print(f'🤖 Agent: {answer}')

### 💡 สังเกต
- Agent ตอบตาม `instruction` ที่เรากำหนด
- แต่ Agent ยังตอบจาก **ความรู้ทั่วไปของ LLM** เท่านั้น
- ยังไม่ได้ค้นจาก **ข้อมูลของเรา** → ต้องให้ "เครื่องมือ" (Tool)

> 🎯 ต่อไปเราจะสร้าง Tool ให้ Agent ใช้งาน

---
## 🔧 Section 5: Tool — ให้ Agent ใช้เครื่องมือ

### FunctionTool คืออะไร?

เขียน Python function → Agent **เรียกใช้ได้เอง** เมื่อจำเป็น

> ⚠️ **docstring สำคัญมาก!** Agent อ่าน docstring เพื่อตัดสินใจว่าจะเรียกใช้ tool ไหน

In [ ]:
# สร้าง Custom Tool
from google.adk.tools import FunctionTool

def calculate_bmi(weight_kg: float, height_cm: float) -> str:
    """คำนวณค่า BMI จากน้ำหนัก (กิโลกรัม) และส่วนสูง (เซนติเมตร)
    
    Args:
        weight_kg: น้ำหนักเป็นกิโลกรัม
        height_cm: ส่วนสูงเป็นเซนติเมตร
    """
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5: status = 'น้ำหนักต่ำกว่าเกณฑ์'
    elif bmi < 25: status = 'น้ำหนักปกติ'
    elif bmi < 30: status = 'น้ำหนักเกิน'
    else: status = 'อ้วน'
    return f'BMI = {bmi:.1f} ({status})'

bmi_tool = FunctionTool(calculate_bmi)

# สร้าง Agent ที่มี Tool
health_agent = LlmAgent(
    name='health_assistant',
    model='gemini-2.5-flash',
    instruction='คุณเป็นผู้ช่วยด้านสุขภาพ ตอบเป็นภาษาไทย ใช้ tool calculate_bmi เมื่อผู้ใช้บอกน้ำหนักและส่วนสูง',
    tools=[bmi_tool]
)

print('✅ สร้าง Health Agent + BMI Tool สำเร็จ')

In [ ]:
# ทดลองใช้ Agent + Tool
answer = await chat_with_agent(health_agent, 'ผมหนัก 70 กก. สูง 175 ซม. BMI เท่าไร?')
print(f'❓ คำถาม: ผมหนัก 70 กก. สูง 175 ซม.')
print(f'🤖 Agent: {answer}')

### 💡 สังเกต
- Agent **อ่าน docstring** แล้วตัดสินใจเรียก `calculate_bmi` เอง
- ไม่ต้อง hard-code: ถ้าถามเรื่องอื่น Agent จะไม่เรียก tool
- **docstring ชัดเจน = Agent เรียกใช้ถูกต้อง**

> 🎯 ต่อไปเราจะสร้าง **RAG Tool** ที่ค้นจาก Qdrant!

---
## 🔍 Section 6: RAG Agent ⭐ — Agent + VectorDB

### หัวใจของ Workshop นี้!

เชื่อม Agent + Qdrant Pipeline จาก Part 1

```
คำถาม → Agent ตัดสินใจ → เรียก search_knowledge
→ ค้นจาก Qdrant → ส่ง context กลับ → Agent สร้างคำตอบ
```

In [ ]:
# สร้าง RAG Tool — ค้นหาจาก Qdrant
def search_knowledge(query: str) -> str:
    """ค้นหาข้อมูลจากฐานความรู้ Case Study ด้าน AI ในประเทศไทย
    ใช้เมื่อต้องการหาข้อมูลเกี่ยวกับ AI ในสาขาต่างๆ เช่น การแพทย์ การเงิน การเกษตร การศึกษา
    
    Args:
        query: คำถามหรือหัวข้อที่ต้องการค้นหา
    """
    results = search_qdrant(query, top_k=3)
    context = '\n\n'.join([f'[{i+1}] {r.payload["text"]}' for i, r in enumerate(results)])
    return f'ผลการค้นหา {len(results)} รายการ:\n{context}'

rag_tool = FunctionTool(search_knowledge)

# สร้าง RAG Agent
rag_agent = LlmAgent(
    name='rag_assistant',
    model='gemini-2.5-flash',
    instruction="""คุณเป็น AI Assistant ที่ตอบคำถามเกี่ยวกับ AI ในประเทศไทย

กฎ:
1. ใช้ tool search_knowledge ค้นหาข้อมูลก่อนตอบเสมอ
2. ตอบจากข้อมูลที่ค้นพบเท่านั้น ห้ามแต่งเอง
3. ถ้าไม่พบข้อมูล ให้บอกตรงๆ ว่าไม่มีข้อมูล
4. ตอบเป็นภาษาไทย สั้นกระชับ""",
    tools=[rag_tool]
)

print('✅ สร้าง RAG Agent สำเร็จ!')

In [ ]:
# ─── ทดลอง RAG Agent ───
questions = [
    'AI ช่วยงานในโรงพยาบาลอย่างไร?',
    'คณะ ICT มหิดล มีโครงการ AI อะไรบ้าง?',
    'ธนาคารใช้ AI อย่างไร?'
]

for q in questions:
    answer = await chat_with_agent(rag_agent, q)
    print(f'\n{"="*60}')
    print(f'❓ {q}')
    print(f'🤖 {answer}')

### 💡 สังเกต
- Agent **ตัดสินใจเอง** ว่าจะค้นหาจาก Qdrant
- คำตอบมาจาก **ข้อมูลจริง** ไม่ใช่แค่ความรู้ทั่วไปของ LLM
- ต่างจาก RAG ธรรมดา: **Agent เลือกได้** ว่าจะค้นหรือตอบเลย

> 🎯 นี่คือ **Agentic RAG** — ต่อไปเราจะเพิ่ม Memory ให้จำบทสนทนาได้!

---
## 🚀 Section 7: Agentic RAG + Memory ⭐

### เพิ่ม Session Memory

ให้ Agent **จำบทสนทนา** ได้ → ถามต่อเนื่องไม่ต้องอธิบายซ้ำ

In [ ]:
# ─── Agentic RAG + Memory ───
from google.adk.sessions import InMemorySessionService

# ใช้ RAG Agent เดิม แต่เพิ่ม session ให้จำได้
runner = InMemoryRunner(agent=rag_agent, app_name='agentic_rag')

async def chat_with_memory(runner, session_id, message):
    content = genai_types.Content(
        role='user',
        parts=[genai_types.Part(text=message)]
    )
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session_id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text

# สร้าง session
session = await runner.session_service.create_session(
    app_name='agentic_rag', user_id='student'
)

# ถามต่อเนื่อง 3 คำถาม
conversations = [
    'AI ในการเกษตรไทยมีอะไรบ้าง?',
    'มีการใช้ NLP ยังไง?',           # ไม่ระบุสาขา — Agent ต้องจำว่ากำลังพูดเรื่องเกษตร
    'ความท้าทายคืออะไร?'            # ต่อเนื่องจากเรื่องเดิม
]

for q in conversations:
    answer = await chat_with_memory(runner, session.id, q)
    print(f'\n{"="*60}')
    print(f'❓ {q}')
    print(f'🤖 {answer}')

### 💡 สังเกต
- คำถามที่ 2 และ 3 **ไม่ระบุว่าเรื่องอะไร** แต่ Agent จำได้ว่ากำลังคุยเรื่องเกษตร
- นี่คือพลังของ **Session Memory** — ทำให้ Agent คุยต่อเนื่องได้
- **Agentic RAG = Agent + RAG + Memory** → ระบบที่ค้นหา + คิด + จำได้

> 🎯 ถ้ารวมทุกอย่าง: Data Pipeline + Agent + Tool + RAG + Memory = **Agentic RAG เต็มรูปแบบ!**

---
## 📋 สรุป: สิ่งที่เราเรียนรู้วันนี้

| Part | Section | สิ่งที่ทำ |
|------|---------|--------|
| 1 | Chunking | ตัดข้อความด้วย Recursive Splitter |
| 1 | Embedding | แปลงเป็น Vector (multilingual-e5-large) |
| 1 | Qdrant | เก็บ + ค้นหา Vector |
| 2 | Agent | สร้าง Agent ด้วย Google ADK |
| 2 | Tool | ให้ Agent ใช้ FunctionTool |
| 2 | RAG Agent ⭐ | Agent ค้นจาก VectorDB |
| 2 | Agentic RAG ⭐ | Agent + RAG + Memory |

### 🔑 Key Takeaways

1. **คุณภาพข้อมูลสำคัญที่สุด** — "Garbage in, garbage out"
2. **Agent ≠ Chatbot** — Agent คิด + ใช้เครื่องมือได้
3. **Agentic RAG** = Agent ที่ตัดสินใจเองว่าจะค้นหาหรือตอบเลย

---

### 🧪 แบบฝึกหัด (1 ชั่วโมง)

เปิด **`agentic_rag_4hr_homework.ipynb`** แล้วทำตามคำชี้แจง (10 คะแนน)

> _"The best way to learn AI is to build with AI."_